# 02 - Missing Value Handling




## Setup

In [1]:
import pandas as pd
from pathlib import Path

INPUT_PATH = Path('../../../data/processed/orders_aggregated.csv')
OUTPUT_PATH = Path('../../../data/processed/orders_clean.csv')

orders_df = pd.read_csv(INPUT_PATH)
print(f"Shape: {orders_df.shape[0]:,} rows x {orders_df.shape[1]} columns")
orders_df.isnull().sum()[orders_df.isnull().sum() > 0]


Shape: 65,752 rows x 20 columns


Order Zipcode    57482
dtype: int64

## 1. Resolve `Order Zipcode` - confirm whether missingness is tied to country

EDA flagged this as likely systemic (86.24% missing at line item level) rather than random. Confirmation before deciding drop vs. flag.


In [7]:
zipcode_missing_by_country = orders_df.groupby('Order Country')['Order Zipcode'].apply(lambda x: x.notna().mean()).sort_values()
print("Fraction of orders WITH a zipcode, by country (lowest first):")
zipcode_missing_by_country.head(15)


Fraction of orders WITH a zipcode, by country (lowest first):


Order Country
Afganistán      0.0
Albania         0.0
Alemania        0.0
Angola          0.0
Arabia Saudí    0.0
Argelia         0.0
Argentina       0.0
Armenia         0.0
Australia       0.0
Austria         0.0
Azerbaiyán      0.0
Bangladés       0.0
Barbados        0.0
Baréin          0.0
Belice          0.0
Name: Order Zipcode, dtype: float64

**What we found:**

**Confirmed: `Order Zipcode` missingness is entirely systemic, tied to country, not random.** 57,482 of 65,752 orders (87.42%) are missing a zipcode - consistent with the 86.24% found at line item level in EDA. Looking at the lowest availability countries (Afghanistan, Albania, Germany, Angola, Saudi Arabia, Algeria, Argentina, Armenia, Australia, Austria, Azerbaijan, Bangladesh, Barbados, Bahrain, Belize), **every single one shows exactly 0.0% zipcode availability** - meaning orders shipped to these countries never carry a zipcode at all, not even occasionally. This is a hard, structural pattern (almost certainly because zipcodes only apply to a small number of countries in this dataset, likely just the US/similar), not noisy or partial missingness.

**Decision confirmed: drop `Order Zipcode` entirely** (rather than keeping it as a binary flag). Given that the vast majority of countries show 0% availability outright, a "has_zipcode" flag would end up being almost equivalent to "is this order going to [the one or two countries where zipcodes exist]" - which is redundant information already captured by `Order Country` itself. Dropping is the cleaner choice here.


In [8]:
DROP_ORDER_ZIPCODE = True  

if DROP_ORDER_ZIPCODE:
    orders_df = orders_df.drop(columns=['Order Zipcode'], errors='ignore')
    print("Dropped Order Zipcode.")
else:
    orders_df['has_zipcode'] = orders_df['Order Zipcode'].notna().astype(int)
    orders_df = orders_df.drop(columns=['Order Zipcode'], errors='ignore')
    print("Converted Order Zipcode to a binary has_zipcode flag.")


Dropped Order Zipcode.


## 2. Handle any remaining negligible missing values

In [9]:
remaining_missing = orders_df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
print("Remaining columns with missing values:")
print(remaining_missing)

if len(remaining_missing) > 0:
    before = len(orders_df)
    orders_df = orders_df.dropna(subset=remaining_missing.index.tolist())
    print(f"\nDropped {before - len(orders_df)} rows with negligible missing values.")


Remaining columns with missing values:
Series([], dtype: int64)


**What we found:**

**Zero remaining missing values after dropping `Order Zipcode` - no rows needed to be dropped for negligible missingness.** This makes sense: `Customer Lname` and `Customer Zipcode` (the two negligible missing fields identified in EDA) were never carried into the aggregated dataset in the first place, since they weren't included in Notebook 01's aggregation rules (they're customer identity fields, not used as model inputs per `docs/data_dictionary.md`). So there was nothing left to clean here beyond the zipcode drop.


## 3. Final missing value check and save

In [11]:
assert orders_df.isnull().sum().sum() == 0, "Missing values remain — check above steps."
print("No missing values remain.")

No missing values remain.


In [12]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
orders_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to: {OUTPUT_PATH.resolve()}")
print(f"Final shape: {orders_df.shape[0]:,} rows x {orders_df.shape[1]} columns")

Saved to: C:\Users\Ewis\Documents\Machine_learning_project\Heads-Up_IT3091\data\processed\orders_clean.csv
Final shape: 65,752 rows x 19 columns


**`DECISION_LOG.md`:** final call on `Order Zipcode` (dropped vs. flagged), with the country level evidence from Section 1.
